# PhyloP Conservation Analysis v2 - Transcript-Aware Flanking Regions

**Key improvement**: Handles spliced transcripts properly - flanking regions come from exonic sequence, not introns.

## Configuration

In [1]:
from pathlib import Path

BIGBED_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/riboseq_orfs/data/Ribo-seq_ORFs.bb"
BIGBED_PATH = Path("data/Ribo-seq_ORFs.bb")

PHYLOP_PATHS = {
    '30way': Path('data/phylop/hg38.phyloP30way.bw'),
    '100way': Path('data/phylop/hg38.phyloP100way.bw'),
    '470way': Path('data/phylop/hg38.phyloP470way.bw')
}

FLANK_SIZE = 50
OUTPUT_PREFIX = 'results/phylop/translon_phylop_v2'

## Import Dependencies

In [2]:
import pyBigWig
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path

## Extract Translon Coordinates WITH Block Information

In [4]:
def extract_translons_from_bigbed(bigbed_path):
    bed_path = str(bigbed_path).replace('.bb', '.bed')
    subprocess.run(['/nfs/production/flicek/ensembl/genebuild/jackt/projects/translon-conservation/bigBedToBed', str(bigbed_path), bed_path], check=True)
    
    translons = pd.read_csv(
        bed_path, 
        sep='\t', 
        header=None,
        usecols=[0, 1, 2, 3, 5, 9, 10, 11],
        names=['chrom', 'start', 'end', 'name', 'strand', 'blockCount', 'blockSizes', 'blockStarts'],
        dtype={'chrom': str, 'start': int, 'end': int, 'name': str, 'strand': str, 
               'blockCount': int, 'blockSizes': str, 'blockStarts': str}
    )
    
    return translons

translons = extract_translons_from_bigbed(BIGBED_PATH)
print(f"Extracted {len(translons):,} translons")
print(f"Multi-exon: {(translons['blockCount'] > 1).sum():,}")
print(f"Single-exon: {(translons['blockCount'] == 1).sum():,}")
translons.head()

Extracted 7,264 translons
Multi-exon: 2,463
Single-exon: 4,801


,chrom,start,end,name,strand,blockCount,blockSizes,blockStarts
0,chr1,826870,829027,c1norep1,+,2,"53,25,","0,2132,"
1,chr1,829023,829092,c1norep2,+,1,"69,","0,"
2,chr1,847671,850321,c1riboseqorf1,+,2,"135,141,","0,2509,"
3,chr1,847671,852067,c1riboseqorf2,+,2,"135,141,","0,4255,"
4,chr1,852070,852699,c1riboseqorf3,+,2,"40,29,","0,600,"


## Core Function: Extract Transcript-Aware Flanks

For spliced features:
- Upstream flank: Last N bp of PREVIOUS exon (or genomic if first exon)
- Downstream flank: First N bp of NEXT exon (or genomic if last exon)

This requires the parent transcript structure, which we don't have. So we'll use a simpler approach:
- Upstream: N bp before the FIRST exon
- Downstream: N bp after the LAST exon

In [6]:
def get_exon_blocks(start, blockCount, blockSizes, blockStarts):
    """Parse BED block notation into list of (start, end) tuples"""
    if blockCount == 1:
        return None
    
    sizes = [int(x) for x in blockSizes.rstrip(',').split(',')]
    starts = [int(x) for x in blockStarts.rstrip(',').split(',')]
    
    blocks = []
    for size, block_start in zip(sizes, starts):
        block_abs_start = start + block_start
        block_abs_end = block_abs_start + size
        blocks.append((block_abs_start, block_abs_end))
    
    return blocks


def get_phylop_scores_transcript_aware(chrom, start, end, strand, blockCount, blockSizes, blockStarts, bw, flank_size=100):
    """Extract PhyloP scores for feature and transcript-aware flanking regions"""
    
    # Extract feature scores from exonic regions only
    feature_scores = []
    
    blocks = get_exon_blocks(start, blockCount, blockSizes, blockStarts)
    
    if blocks is None:
        # Single exon - simple case
        feature_scores = bw.values(chrom, start, end, numpy=True)
        feature_scores = feature_scores[~np.isnan(feature_scores)] if feature_scores is not None else np.array([])
        first_exon_start = start
        last_exon_end = end
    else:
        # Multi-exon - extract from each block
        for block_start, block_end in blocks:
            block_scores = bw.values(chrom, block_start, block_end, numpy=True)
            if block_scores is not None:
                block_scores = block_scores[~np.isnan(block_scores)]
                feature_scores.extend(block_scores)
        
        feature_scores = np.array(feature_scores)
        first_exon_start = blocks[0][0]
        last_exon_end = blocks[-1][1]
    
    # Get flanking regions relative to exon boundaries
    # Upstream: before first exon
    # Downstream: after last exon
    upstream_scores = bw.values(chrom, max(0, first_exon_start - flank_size), first_exon_start, numpy=True)
    downstream_scores = bw.values(chrom, last_exon_end, last_exon_end + flank_size, numpy=True)
    
    upstream_scores = upstream_scores[~np.isnan(upstream_scores)] if upstream_scores is not None else np.array([])
    downstream_scores = downstream_scores[~np.isnan(downstream_scores)] if downstream_scores is not None else np.array([])
    
    return upstream_scores, feature_scores, downstream_scores

## Metric Calculation Functions

In [7]:
def max_run_length(boolean_array):
    if len(boolean_array) == 0:
        return 0
    max_run = current_run = 0
    for val in boolean_array:
        if val:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 0
    return max_run


def calculate_nucleotide_metrics(scores, prefix=''):
    if len(scores) == 0:
        return {f'{prefix}n_bases': 0}
    
    return {
        f'{prefix}n_bases': len(scores),
        f'{prefix}mean': np.mean(scores),
        f'{prefix}median': np.median(scores),
        f'{prefix}std': np.std(scores),
        f'{prefix}min': np.min(scores),
        f'{prefix}max': np.max(scores),
        f'{prefix}q25': np.percentile(scores, 25),
        f'{prefix}q75': np.percentile(scores, 75),
        f'{prefix}frac_positive': np.sum(scores > 0) / len(scores),
        f'{prefix}frac_conserved_weak': np.sum(scores > 0.5) / len(scores),
        f'{prefix}frac_conserved_strong': np.sum(scores > 1.5) / len(scores),
        f'{prefix}frac_negative': np.sum(scores < 0) / len(scores),
        f'{prefix}frac_depleted': np.sum(scores < -0.5) / len(scores),
        f'{prefix}max_conserved_run': max_run_length(scores > 0),
    }


def calculate_codon_metrics(scores, strand):
    if len(scores) == 0:
        return {'n_codons': 0}
    
    if strand == '-':
        scores = scores[::-1]
    
    n_codons = len(scores) // 3
    if n_codons == 0:
        return {'n_codons': 0}
    
    trimmed_scores = scores[:n_codons * 3]
    scores_2d = trimmed_scores.reshape(-1, 3)
    
    return {
        'n_codons': n_codons,
        'codon_pos1_mean': np.mean(scores_2d[:, 0]),
        'codon_pos2_mean': np.mean(scores_2d[:, 1]),
        'codon_pos3_mean': np.mean(scores_2d[:, 2]),
        'wobble_depletion_mean': (np.mean(scores_2d[:, 0]) + np.mean(scores_2d[:, 1])) / 2 - np.mean(scores_2d[:, 2]),
        'codon_pos1_frac_conserved': np.sum(scores_2d[:, 0] > 1.5) / n_codons,
        'codon_pos2_frac_conserved': np.sum(scores_2d[:, 1] > 1.5) / n_codons,
        'codon_pos3_frac_conserved': np.sum(scores_2d[:, 2] > 1.5) / n_codons,
    }


def calculate_flanking_context(feature_scores, upstream_scores, downstream_scores):
    if len(feature_scores) == 0:
        return {}
    
    feature_mean = np.mean(feature_scores)
    metrics = {}
    
    if len(upstream_scores) > 0:
        upstream_mean = np.mean(upstream_scores)
        metrics['upstream_mean'] = upstream_mean
        metrics['feature_vs_upstream_diff'] = feature_mean - upstream_mean
    else:
        metrics.update({'upstream_mean': np.nan, 'feature_vs_upstream_diff': np.nan})
    
    if len(downstream_scores) > 0:
        downstream_mean = np.mean(downstream_scores)
        metrics['downstream_mean'] = downstream_mean
        metrics['feature_vs_downstream_diff'] = feature_mean - downstream_mean
    else:
        metrics.update({'downstream_mean': np.nan, 'feature_vs_downstream_diff': np.nan})
    
    if len(upstream_scores) > 0 and len(downstream_scores) > 0:
        flanking_mean = np.mean(np.concatenate([upstream_scores, downstream_scores]))
        metrics['flanking_mean'] = flanking_mean
        metrics['feature_vs_flanking_diff'] = feature_mean - flanking_mean
        metrics['conservation_specificity'] = feature_mean - max(upstream_mean, downstream_mean)
        metrics['specifically_conserved'] = (feature_mean - flanking_mean) > 0.5
    else:
        metrics.update({
            'flanking_mean': np.nan,
            'feature_vs_flanking_diff': np.nan,
            'conservation_specificity': np.nan,
            'specifically_conserved': False
        })
    
    return metrics

## Main Analysis Function

In [8]:
def analyze_translons_phylop(translons_df, phylop_paths, flank_size=100):
    all_results = []
    
    for phylop_name, phylop_path in phylop_paths.items():
        print(f"\nProcessing {phylop_name}...")
        if not phylop_path.exists():
            print(f"  File not found: {phylop_path}")
            continue
            
        bw = pyBigWig.open(str(phylop_path))
        
        for idx, translon in translons_df.iterrows():
            if idx % 1000 == 0:
                print(f"  {idx}/{len(translons_df)}...")
            
            try:
                upstream_scores, feature_scores, downstream_scores = get_phylop_scores_transcript_aware(
                    translon['chrom'], 
                    int(translon['start']), 
                    int(translon['end']), 
                    translon['strand'],
                    int(translon['blockCount']),
                    translon['blockSizes'],
                    translon['blockStarts'],
                    bw, 
                    flank_size
                )
                
                if len(feature_scores) == 0:
                    continue
                
                result = {
                    'translon_id': translon['name'],
                    'chrom': translon['chrom'],
                    'start': int(translon['start']),
                    'end': int(translon['end']),
                    'strand': translon['strand'],
                    'blockCount': int(translon['blockCount']),
                    'exonic_length': len(feature_scores),
                    'genomic_span': int(translon['end']) - int(translon['start']),
                    'phylop_dataset': phylop_name,
                    'flank_size': flank_size,
                }
                
                result.update(calculate_nucleotide_metrics(feature_scores, prefix='feature_'))
                result.update(calculate_nucleotide_metrics(upstream_scores, prefix='upstream_'))
                result.update(calculate_nucleotide_metrics(downstream_scores, prefix='downstream_'))
                result.update(calculate_codon_metrics(feature_scores, translon['strand']))
                result.update(calculate_flanking_context(feature_scores, upstream_scores, downstream_scores))
                
                all_results.append(result)
                
            except Exception as e:
                continue
        
        bw.close()
        print(f"  Completed: {len([r for r in all_results if r['phylop_dataset'] == phylop_name])} translons")
    
    return pd.DataFrame(all_results)

## Run Analysis

In [9]:
Path(OUTPUT_PREFIX).parent.mkdir(parents=True, exist_ok=True)

available_phylop = {name: path for name, path in PHYLOP_PATHS.items() if path.exists()}
print(f"Found {len(available_phylop)} PhyloP dataset(s): {list(available_phylop.keys())}")

results = analyze_translons_phylop(translons, available_phylop, FLANK_SIZE)

output_file = f"{OUTPUT_PREFIX}_comprehensive.tsv"
results.to_csv(output_file, sep='\t', index=False)
print(f"\nSaved {len(results):,} results to {output_file}")

Found 3 PhyloP dataset(s): ['30way', '100way', '470way']

Processing 30way...
  0/7264...
  1000/7264...
  2000/7264...
  3000/7264...
  4000/7264...
  5000/7264...
  6000/7264...
  7000/7264...
  Completed: 7262 translons

Processing 100way...
  0/7264...
  1000/7264...
  2000/7264...
  3000/7264...
  4000/7264...
  5000/7264...
  6000/7264...
  7000/7264...
  Completed: 7257 translons

Processing 470way...
  0/7264...
  1000/7264...
  2000/7264...
  3000/7264...
  4000/7264...
  5000/7264...
  6000/7264...
  7000/7264...
  Completed: 7263 translons

Saved 21,782 results to results/phylop/translon_phylop_v2_comprehensive.tsv


## Summary Statistics

In [10]:
for dataset in results['phylop_dataset'].unique():
    df = results[results['phylop_dataset'] == dataset]
    
    print(f"\n{'='*80}")
    print(f"Dataset: {dataset}")
    print(f"{'='*80}")
    print(f"Translons: {len(df):,}")
    print(f"\nFeature conservation:")
    print(f"  Mean: {df['feature_mean'].mean():.3f} ± {df['feature_mean'].std():.3f}")
    print(f"  Median: {df['feature_median'].median():.3f}")
    
    print(f"\nFlanking context:")
    print(f"  Feature vs flanking diff: {df['feature_vs_flanking_diff'].mean():.3f}")
    print(f"  Specifically conserved: {df['specifically_conserved'].sum():,} ({100*df['specifically_conserved'].sum()/len(df):.1f}%)")
    
    print(f"\nConservation specificity (feature - max(flanks)):")
    print(f"  >1.0: {(df['conservation_specificity'] > 1.0).sum():,} ({100*(df['conservation_specificity'] > 1.0).sum()/len(df):.1f}%)")
    print(f"  >0.5: {(df['conservation_specificity'] > 0.5).sum():,} ({100*(df['conservation_specificity'] > 0.5).sum()/len(df):.1f}%)")
    print(f"  >0.0: {(df['conservation_specificity'] > 0.0).sum():,} ({100*(df['conservation_specificity'] > 0.0).sum()/len(df):.1f}%)")
    print(f"  <0.0: {(df['conservation_specificity'] < 0.0).sum():,} ({100*(df['conservation_specificity'] < 0.0).sum()/len(df):.1f}%)")


Dataset: 30way
Translons: 7,262

Feature conservation:
  Mean: 0.317 ± 0.343
  Median: 0.205

Flanking context:
  Feature vs flanking diff: 0.012
  Specifically conserved: 60 (0.8%)

Conservation specificity (feature - max(flanks)):
  >1.0: 0 (0.0%)
  >0.5: 9 (0.1%)
  >0.0: 2,046 (28.2%)
  <0.0: 5,215 (71.8%)

Dataset: 100way
Translons: 7,257

Feature conservation:
  Mean: 0.879 ± 1.417
  Median: 0.257

Flanking context:
  Feature vs flanking diff: 0.044
  Specifically conserved: 1,102 (15.2%)

Conservation specificity (feature - max(flanks)):
  >1.0: 229 (3.2%)
  >0.5: 518 (7.1%)
  >0.0: 2,050 (28.2%)
  <0.0: 5,204 (71.7%)

Dataset: 470way
Translons: 7,263

Feature conservation:
  Mean: 1.560 ± 2.408
  Median: 0.492

Flanking context:
  Feature vs flanking diff: 0.093
  Specifically conserved: 1,956 (26.9%)

Conservation specificity (feature - max(flanks)):
  >1.0: 553 (7.6%)
  >0.5: 968 (13.3%)
  >0.0: 2,025 (27.9%)
  <0.0: 5,237 (72.1%)


## Find Specifically Conserved Candidates

In [11]:
# High confidence: strong feature + weak flanks
for dataset in results['phylop_dataset'].unique():
    df = results[results['phylop_dataset'] == dataset]
    
    high_conf = df[
        (df['feature_mean'] > 1.5) &
        (df['conservation_specificity'] > 1.0)
    ].sort_values('conservation_specificity', ascending=False)
    
    print(f"\n{'='*80}")
    print(f"{dataset}: High confidence candidates (feature >1.5, specificity >1.0)")
    print(f"{'='*80}")
    print(f"Count: {len(high_conf)}\n")
    
    if len(high_conf) > 0:
        print(high_conf[['translon_id', 'chrom', 'exonic_length', 'blockCount',
                         'feature_mean', 'upstream_mean', 'downstream_mean',
                         'conservation_specificity']].head(20).to_string(index=False))


30way: High confidence candidates (feature >1.5, specificity >1.0)
Count: 0


100way: High confidence candidates (feature >1.5, specificity >1.0)
Count: 221

     translon_id chrom  exonic_length  blockCount  feature_mean  upstream_mean  downstream_mean  conservation_specificity
 c15riboseqorf40 chr15             54           1      7.583389        2.01830          2.32274                  5.260649
     c10norep101 chr10             69           1      5.598057        1.44630          1.47354                  4.124517
       c1norep57  chr1            174           2      4.926552        1.02382          0.89814                  3.902732
c14riboseqorf117 chr14            294           2      6.306854        2.59024          1.18276                  3.716614
       c4norep39  chr4            246           3      5.180171        1.44170          1.70888                  3.471291
 c2riboseqorf136  chr2            384           1      4.183336        0.80686          0.41192              